# Explore the dataset and try a prediction

This beginner-friendly notebook shows:

1. How many images each class has in `dataset/train`, `dataset/validation` and `dataset/test`
2. A few sample images per class
3. How to run a single-image prediction with the trained MobileNetV3 model

**Before running the cells** (from the project root, in the VS Code terminal):

```powershell
py -m pip install -r requirements.txt   # install dependencies (once)
py scripts\prepare_dataset.py           # prepare the dataset (once)
py src\train_mobilenet.py               # train the main model (once)
```

In VS Code, select your Python environment (top-right corner) and run the cells with `Shift+Enter`.

In [ ]:
# Make the project importable no matter where Jupyter starts.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

print("Project root:", PROJECT_ROOT)
print("Dataset dir :", config.DATASET_DIR)

In [ ]:
# 1) Count the images per class and split (no TensorFlow needed).
try:
    config.require_dataset()
    for split in ("train", "validation", "test"):
        print(f"{split}:")
        for class_name in config.CLASSES:
            class_dir = config.DATASET_DIR / split / class_name
            count = len(list(class_dir.glob("*.jpg"))) + len(list(class_dir.glob("*.png")))
            print(f"  {class_name:35s} {count:5d} images")
except config.DatasetNotFoundError as error:
    print(error)

In [ ]:
# 2) Show a few sample images per class.
import matplotlib.pyplot as plt
from PIL import Image

try:
    config.require_dataset()
    fig, axes = plt.subplots(len(config.CLASSES), 3, figsize=(9, 3 * len(config.CLASSES)))
    for row, class_name in enumerate(config.CLASSES):
        samples = sorted((config.TRAIN_DIR / class_name).glob("*.jpg"))[:3]
        for column in range(3):
            axis = axes[row][column] if len(config.CLASSES) > 1 else axes[column]
            axis.axis("off")
            if column < len(samples):
                axis.imshow(Image.open(samples[column]))
                if column == 0:
                    axis.set_title(config.disease_name(class_name), fontsize=10, loc="left")
    fig.tight_layout()
    plt.show()
except config.DatasetNotFoundError as error:
    print(error)

In [ ]:
# 3) Predict one test image with the trained MobileNetV3 model.
from src import inference

try:
    config.require_dataset()
    sample_image = sorted((config.TEST_DIR / config.CLASSES[1]).glob("*.jpg"))[0]
    print("Predicting:", sample_image)

    result = inference.predict_image(sample_image)
    print(inference.format_prediction(result))
except config.DatasetNotFoundError as error:
    print(error)
except FileNotFoundError as error:
    # Model or image missing - the message explains what to run first.
    print(error)